# 01 — Insights comerciales sobre cartera AFAP

EDA sobre el dataset Bank Customer Churn (Kaggle, 10.000 clientes) reframeado al vocabulario AFAP.

**Las 3 preguntas del equipo comercial:**
1. ¿A quién se me va a ir? — scoring de fuga por afiliado.
2. ¿Dónde concentrar el foco? — Pareto de cartera.
3. ¿Qué está cambiando? — variación entre segmentos y cohortes.

Números reales medidos en este notebook:
```
N = 10.000 afiliados
Saldo total:        UYU 764,9 M
Ticket promedio:    UYU 76.486
Tasa de fuga:       20,38 %
Pct. activos:       51,51 %
Top 20%:            40,2 % del saldo
Top 50%:            85,7 % del saldo
Canelones:          32,4 % de fuga  (2× la media)
Inactivos:          26,9 %  vs Activos 14,3 %
Edad 45-65:         ~48-50 % de fuga  (pico)
Edad <35:           7,9 %
```

In [1]:
import sys
from pathlib import Path

# Permitir `from src. ...` cuando se corre el notebook desde notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data_loader import cargar_csv
from src.analytics import (
    kpis_globales,
    pareto_cartera,
    concentracion_cartera,
    aplicar_segmentaciones,
    tasa_fuga_por_segmento,
    heatmap_edad_antiguedad,
    variacion_entre_segmentos,
)
from src.models.churn_logit import entrenar_logit, scorear_afiliados, drivers_principales

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [2]:
df = cargar_csv(ROOT / 'data' / 'raw' / 'Customer-Churn-Records.csv')
print(df.shape)
df.head()

(10000, 13)


,afiliado_id,apellido,score_interno,departamento,genero,edad,anios_afiliacion,saldo_cuenta,productos_contratados,tiene_producto_adicional,aportante_activo,salario_nominal,traspaso
0,15634602,Hargrave,619,Montevideo,F,42,2,0.0000,1,1,1,"101,348.8800",1
1,15647311,Hill,608,Maldonado,F,41,1,"83,807.8600",1,0,1,"112,542.5800",0
2,15619304,Onio,502,Montevideo,F,42,8,"159,660.8000",3,1,0,"113,931.5700",1
3,15701354,Boni,699,Montevideo,F,39,1,0.0000,2,0,0,"93,826.6300",0
4,15737888,Mitchell,850,Maldonado,F,43,2,"125,510.8200",1,1,1,"79,084.1000",0


## KPIs globales

In [3]:
kpis_globales(df)

{'n_afiliados': 10000,
 'saldo_total': 764858892.88,
 'ticket_promedio': 76485.889288,
 'tasa_fuga': 0.2038,
 'pct_activos': 0.5151}

## 1) Pareto de cartera — regla 80/20

In [4]:
p = pareto_cartera(df)
print(f'Top 20% concentra: {concentracion_cartera(df, 0.20)*100:.1f} %')
print(f'Top 50% concentra: {concentracion_cartera(df, 0.50)*100:.1f} %')
p.head(10)

Top 20% concentra: 40.2 %
Top 50% concentra: 85.7 %


,afiliado_id,saldo_cuenta,rank,pct_acumulado,pct_afiliados,top_80
0,15757408,"250,898.0900",1,0.0003,0.0001,True
1,15715622,"238,387.5600",2,0.0006,0.0002,True
2,15714241,"222,267.6300",3,0.0009,0.0003,True
3,15571958,"221,532.8000",4,0.0012,0.0004,True
4,15586674,"216,109.8800",5,0.0015,0.0005,True
5,15599131,"214,346.9600",6,0.0018,0.0006,True
6,15594408,"213,146.2000",7,0.0021,0.0007,True
7,15769818,"212,778.2000",8,0.0023,0.0008,True
8,15620268,"212,696.3200",9,0.0026,0.0009,True
9,15780212,"212,692.9700",10,0.0029,0.0010,True


## 2) Perfil de fuga — departamento, actividad y edad

In [5]:
tasa_fuga_por_segmento(df, 'departamento')

,departamento,n_afiliados,n_fugas,saldo_total,tasa_fuga
0,Canelones,2509,814,"300,402,861.3800",0.3244
1,Maldonado,2477,413,"153,123,552.0100",0.1667
2,Montevideo,5014,811,"311,332,479.4900",0.1617


In [6]:
tasa_fuga_por_segmento(df, 'aportante_activo')

,aportante_activo,n_afiliados,n_fugas,saldo_total,tasa_fuga
0,0,4849,1303,"374,024,593.4100",0.2687
1,1,5151,735,"390,834,299.4700",0.1427


In [7]:
seg = aplicar_segmentaciones(df)
tasa_fuga_por_segmento(seg, 'tramo_edad')

,tramo_edad,n_afiliados,n_fugas,saldo_total,tasa_fuga
0,55-65,600,299,"48,945,928.7700",0.4983
1,45-55,1458,702,"118,495,028.6800",0.4815
2,35-45,3981,704,"305,449,762.0000",0.1768
3,65+,282,43,"20,216,469.9900",0.1525
4,<35,3679,290,"271,751,703.4400",0.0788


## Heatmap edad × antigüedad (tasa de fuga)

In [8]:
heatmap_edad_antiguedad(df)

tramo_antiguedad,0-2,3-5,6-8,9+
tramo_edad,,,,
<35,0.0722,0.0870,0.0771,0.0759
35-45,0.1840,0.1668,0.1728,0.1931
45-55,0.4948,0.5210,0.4209,0.5000
55-65,0.5000,0.5137,0.4651,0.5281
65+,0.1429,0.1512,0.1600,0.1569


## Variación entre segmentos (vs. baseline global)

In [9]:
variacion_entre_segmentos(df, 'departamento')

,departamento,n_afiliados,n_fugas,saldo_total,tasa_fuga,delta_abs,delta_rel
0,Canelones,2509,814,"300,402,861.3800",0.3244,0.1206,0.5919
1,Maldonado,2477,413,"153,123,552.0100",0.1667,-0.0371,-0.1819
2,Montevideo,5014,811,"311,332,479.4900",0.1617,-0.0421,-0.2063


## 3) Modelo de scoring — logística interpretable

In [10]:
modelo = entrenar_logit(df, seed=42)
drivers_principales(modelo, top_n=8)

,feature,coef,abs_coef,signo,narrativa
0,aportante_activo,-1.0720,1.0720,reduce,'aportante_activo' reduce la probabilidad de fuga
1,departamento_Montevideo,-0.7662,0.7662,reduce,'departamento_Montevideo' reduce la probabilid...
2,edad,0.7613,0.7613,aumenta,'edad' aumenta la probabilidad de fuga
3,departamento_Maldonado,-0.7321,0.7321,reduce,'departamento_Maldonado' reduce la probabilida...
4,genero_M,-0.5252,0.5252,reduce,'genero_M' reduce la probabilidad de fuga
5,saldo_cuenta,0.1662,0.1662,aumenta,'saldo_cuenta' aumenta la probabilidad de fuga
6,score_interno,-0.0636,0.0636,reduce,'score_interno' reduce la probabilidad de fuga
7,productos_contratados,-0.0582,0.0582,reduce,'productos_contratados' reduce la probabilidad...


In [11]:
scores = scorear_afiliados(modelo, df)
top100 = scores.head(100).merge(
    df[['afiliado_id', 'apellido', 'departamento', 'edad', 'saldo_cuenta']],
    on='afiliado_id',
)
print(f'Top 5% ({int(len(df)*0.05)} afiliados) concentra {scores.head(int(len(df)*0.05))["prob_fuga"].sum() / scores["prob_fuga"].sum() * 100:.1f}% del riesgo total')
top100.head(20)

Top 5% (500 afiliados) concentra 16.1% del riesgo total


,afiliado_id,prob_fuga,apellido,departamento,edad,saldo_cuenta
0,15737647,0.9299,Obioma,Canelones,77,"135,120.5600"
1,15653251,0.9263,Hickey,Montevideo,84,"87,873.3900"
2,15653050,0.9206,Norriss,Canelones,76,"95,052.2900"
3,15655360,0.9048,Chikelu,Canelones,72,"148,666.9900"
4,15790113,0.9035,Schofield,Canelones,71,"113,317.1000"
5,15591107,0.8852,Flemming,Canelones,68,"110,357.0000"
6,15794360,0.8785,Hao,Canelones,70,"71,816.7400"
7,15775761,0.8720,Iweobiegbunam,Canelones,69,"86,038.2100"
8,15648967,0.8682,Ch'en,Canelones,64,"169,362.4300"
9,15807889,0.8661,Wood,Canelones,74,"108,891.7000"
